# Travel Analytics Capstone — Project Setup
**Project summary:** This notebook develops one component of a travel analytics system using the supplied CSV datasets. The broader project combines flight-price regression, gender classification, hotel recommendations, a Flask REST API, a Streamlit dashboard, MLflow tracking, Docker, Kubernetes, Airflow and CI/CD.

**GitHub repository:** `https://github.com/YOUR-USERNAME/travel-analytics-capstone`

> Before submission, replace the placeholder above with the one repository URL containing the full project. Keep all three notebooks pointed to the same repository.

In [ ]:
# In Google Colab, upload the CSVs in the Files panel or mount Google Drive.
# If the repository ZIP is uploaded to Colab, unzip it and set DATA_DIR accordingly.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("/content/data")  # Change to the folder containing flights.csv, hotels.csv, users.csv
if not (DATA_DIR / "flights.csv").exists():
    # Local notebook fallback: place CSVs in ./data
    DATA_DIR = Path("data")

flights = pd.read_csv(DATA_DIR / "flights.csv")
hotels = pd.read_csv(DATA_DIR / "hotels.csv")
users = pd.read_csv(DATA_DIR / "users.csv")

print("Flights:", flights.shape)
print("Hotels:", hotels.shape)
print("Users:", users.shape)
display(flights.head())
display(hotels.head())
display(users.head())

## Data understanding and quality checks

The flight dataset contains route, flight type, duration, distance, agency, date and observed price. The hotel dataset contains hotel, place, stay duration, daily price and total price. The users dataset contains age, company and a recorded gender label.

The first checks validate schema, duplicates, missing values, numeric types and date parsing. These checks matter because model quality depends on consistent input definitions; a pipeline that silently changes column types can fail after deployment.

In [ ]:
for name, frame in {"flights": flights, "hotels": hotels, "users": users}.items():
    print(f"\\n{name}: {frame.shape}")
    display(frame.dtypes.rename("dtype").to_frame())
    print("Missing values:", int(frame.isna().sum().sum()))
    print("Duplicate rows:", int(frame.duplicated().sum()))

display(flights.describe(include="all").T)

# 1. Regression — Flight Price Prediction

## Objective and reasoning
The target is `price` in `flights.csv`. Candidate predictors describe the journey: origin, destination, flight type, duration, distance, agency and booking/travel date. Identifier fields (`travelCode`, `userCode`) are excluded because they are identifiers rather than stable journey characteristics and can cause leakage or memorization.

We compare a regularized linear baseline with a random-forest regressor. The same held-out split and metrics are used for a fair comparison. MAE is directly interpretable in source price units, RMSE penalizes large errors more strongly, and R² summarizes variance explained. A random split is useful for a first baseline; a production assessment should additionally use chronological validation to mimic future price prediction.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = flights.copy()
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["price"])
dt = pd.to_datetime(df["date"], errors="coerce")
df["date_year"] = dt.dt.year.fillna(2000)
df["date_month"] = dt.dt.month.fillna(1)
df["date_dayofweek"] = dt.dt.dayofweek.fillna(0)

cat_cols = ["from", "to", "flightType", "agency"]
num_cols = ["time", "distance", "date_year", "date_month", "date_dayofweek"]
X = df[cat_cols + num_cols].copy()
y = df["price"].astype(float)
for col in cat_cols:
    X[col] = X[col].fillna("unknown").astype(str)
for col in num_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
preprocess = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2))
    ]), cat_cols),
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), num_cols)
])
models = {
    "Ridge baseline": Ridge(alpha=2.0),
    "Random forest": RandomForestRegressor(
        n_estimators=180, min_samples_leaf=2, random_state=42, n_jobs=-1
    )
}
results = []
fitted = {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    row = {
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    }
    results.append(row)
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values("MAE")
display(results_df)
best_name = results_df.iloc[0]["model"]
best_model = fitted[best_name]
print("Selected by lowest holdout MAE:", best_name)

## Interpretation and limitations

Use the measured table above to state which model performed better; do not assume that a more complex model must win. Investigate residuals and error by route, agency and price range. If flight prices are recorded in a currency not documented in the CSV, report predictions in **source dataset units** rather than inventing a currency.

Potential leakage must be considered carefully: the date definition should match the real prediction moment. If the target is meant to be predicted before booking, do not include fields only known after the outcome. Evaluate a chronological split before claiming production readiness.

In [ ]:
pred = best_model.predict(X_test)
residuals = y_test.to_numpy() - pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(pred[:3000], residuals[:3000], alpha=0.35)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Predicted price")
axes[0].set_ylabel("Actual - predicted")
axes[0].set_title("Residual sample")
axes[1].hist(residuals, bins=40)
axes[1].set_title("Residual distribution")
axes[1].set_xlabel("Residual")
plt.tight_layout()
plt.show()

In [ ]:
# Save the selected pipeline so the Flask API uses the exact same preprocessing.
import joblib, json
from pathlib import Path
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
joblib.dump(best_model, ARTIFACT_DIR / "flight_price_model.joblib")
(ARTIFACT_DIR / "flight_regression_metrics.json").write_text(
    results_df.to_json(orient="records", indent=2)
)
print("Saved model and metrics to", ARTIFACT_DIR.resolve())

## MLOps handoff
The saved object includes preprocessing and the estimator, so API requests do not need to duplicate training transformations. In the shared GitHub repository, the production training module also logs candidate runs and metrics with MLflow. The Flask endpoint loads this artifact; Docker packages the service; Kubernetes manages replicas; Airflow schedules retraining; Jenkins runs tests and build steps.

**Submission note:** Record the metrics generated in your own run, add a screenshot of the output table and residual plot, and replace YOUR-USERNAME with your GitHub username after creating the repository, using the same URL in all three notebooks.